# Performance analysis

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.calibration import calibration_curve

from practice_sdoml.modeling.train import load_data, train

def evaluate_and_generate_figures():
    # 1. Definir directorio de salida para cumplir con reports/figures/
    # Resolves the current working directory to the project root and points to reports/figures
    reports_dir = Path.cwd().resolve()
    if reports_dir.name in ["notebooks", "modeling"]:
        reports_dir = reports_dir.parent.parent if reports_dir.name == "modeling" else reports_dir.parent

    reports_dir = reports_dir / "reports" / "figures"
    reports_dir.mkdir(parents=True, exist_ok=True)

    # 2. Cargar datos y entrenar el modelo
    train_loader, num_features, num_classes = load_data(batch_size=32)
    epochs = 15
    
    # Entrenamos registrando la pérdida de cada época
    model = train(epochs=epochs)
    model.eval()

    # 3. Obtener predicciones, probabilidades y pérdidas por muestra
    all_preds = []
    all_targets = []
    all_probs = []
    sample_losses = []
    criterion_none = nn.CrossEntropyLoss(reduction="none")

    with torch.no_grad():
        for batch_x, batch_y in train_loader:
            outputs = model(batch_x)
            losses = criterion_none(outputs, batch_y)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            sample_losses.extend(losses.numpy())
            all_preds.extend(preds.numpy())
            all_targets.extend(batch_y.numpy())
            all_probs.extend(probs.numpy())

    all_targets = np.array(all_targets)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    sample_losses = np.array(sample_losses)

    # =========================================================================
    # Figura 1: Confusion Matrix
    # =========================================================================
    plt.figure(figsize=(8, 8))
    cm = confusion_matrix(all_targets, all_preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap="Blues", values_format="d", ax=plt.gca())
    plt.title("Confusion Matrix - Diabetes Risk Model", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(reports_dir / "confusion_matrix.png", dpi=300)
    plt.close()
    print("Guardada: reports/figures/confusion_matrix.png")

    # =========================================================================
    # Figura 2: (Top Highest Loss)[cite: 1]
    # =========================================================================
    plt.figure(figsize=(8, 8))
    top_k = 10
    worst_indices = np.argsort(sample_losses)[-top_k:]
    worst_losses = sample_losses[worst_indices]

    sns.barplot(x=[f"Sample {i}" for i in worst_indices], y=worst_losses, color="crimson")
    plt.title(f"Top {top_k} Samples with Highest Loss (Worst Misclassifications)", fontsize=12, fontweight="bold")
    plt.xlabel("Sample Index", fontsize=10)
    plt.ylabel("Loss Value", fontsize=10)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(reports_dir / "top_loss_samples.png", dpi=300)
    plt.close()
    print("Guardada: reports/figures/top_loss_samples.png")

    # =========================================================================
    # Figura 3: (Calibration Reliability Diagram)[cite: 1]
    # =========================================================================
    plt.figure(figsize=(8, 8))
    # Calibramos la clase positiva (clase 1)
    target_binary = (all_targets == 1).astype(int)
    prob_pos = all_probs[:, 1] if num_classes > 1 else all_probs[:, 0]
    
    fraction_of_positives, mean_predicted_value = calibration_curve(
        target_binary, prob_pos, n_bins=5, strategy="uniform"
    )

    plt.plot(mean_predicted_value, fraction_of_positives, "s-", color="magenta", label="Model Calibration")
    plt.plot([0, 1], [0, 1], "k--", label="Perfect Calibration")  # Diagonal ideal[cite: 1]
    plt.title("Model Calibration Analysis (Reliability Diagram)", fontsize=12, fontweight="bold")
    plt.xlabel("Mean Predicted Probability", fontsize=10)
    plt.ylabel("Fraction of Positives", fontsize=10)
    plt.legend(loc="upper left")
    plt.tight_layout()
    plt.savefig(reports_dir / "calibration_curve.png", dpi=300)
    plt.close()
    print("Guardada: reports/figures/calibration_curve.png")

if __name__ == "__main__":
    evaluate_and_generate_figures()

--- Initializing training ---
Epoch [1/15] - Loss: 3.7768
Epoch [2/15] - Loss: 1.2060
Epoch [3/15] - Loss: 1.3194
Epoch [4/15] - Loss: 1.0560
Epoch [5/15] - Loss: 0.9231
Epoch [6/15] - Loss: 0.9554
Epoch [7/15] - Loss: 1.0271
Epoch [8/15] - Loss: 0.9804
Epoch [9/15] - Loss: 0.9286
Epoch [10/15] - Loss: 0.8503
Epoch [11/15] - Loss: 0.8684
Epoch [12/15] - Loss: 0.7479
Epoch [13/15] - Loss: 0.8084
Epoch [14/15] - Loss: 0.7949
Epoch [15/15] - Loss: 0.7002
--- Training finished ---
Guardada: reports/figures/confusion_matrix.png
Guardada: reports/figures/top_loss_samples.png
Guardada: reports/figures/calibration_curve.png
